In [1]:
# ============================================
#  PHASE 4 — ONE-PAGE XGBOOST TRAINING SCRIPT
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib

# --------------------------------------------
# 1. Load preprocessed dataset
# --------------------------------------------
df = pd.read_csv("preprocessed_creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

# --------------------------------------------
# 2. Train / Validation / Test Split
# --------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# --------------------------------------------
# 3. Handle Class Imbalance (Choose ONE)
# --------------------------------------------

USE_SMOTE = False   # Set to True if you want SMOTE

if USE_SMOTE:
    sm = SMOTE(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)
    print("SMOTE applied:", X_train.shape, y_train.shape)
else:
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_pos_weight = neg / pos
    print("scale_pos_weight:", scale_pos_weight)

# --------------------------------------------
# 4. Train XGBoost Model
# --------------------------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight if not USE_SMOTE else 1,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)
print("Model training complete.")

# --------------------------------------------
# 5. Evaluate Model
# --------------------------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_proba)
print("ROC-AUC:", roc_auc)

# --------------------------------------------
# 6. Save Model
# --------------------------------------------
joblib.dump(model, "xgb_fraud_model.pkl")
print("Model saved to ../models/xgb_fraud_model.pkl")

Train: (38885, 30) Val: (8332, 30) Test: (8333, 30)
scale_pos_weight: 355.74311926605503
Model training complete.

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      8309
         1.0       0.86      0.79      0.83        24

    accuracy                           1.00      8333
   macro avg       0.93      0.90      0.91      8333
weighted avg       1.00      1.00      1.00      8333

ROC-AUC: 0.9642355678581458
Model saved to ../models/xgb_fraud_model.pkl
